# Moteur de recherche — FIFA World Cup

Projet de Programmation de spécialité (Python), **Partie 3 : interface graphique**.

Conformément aux consignes, **seule l'interface** est écrite dans ce notebook : toutes les
classes (`Document`, `Corpus`, `SearchEngine`…) sont importées depuis les modules `.py` du projet.

**Pour utiliser l'interface** : menu *Run All* (ou exécuter les cellules dans l'ordre).

## 1. Préparation : réutiliser le moteur

### 1.1 Import des classes et chargement du corpus
Le corpus nettoyé est rechargé depuis `data/corpus_sauvegarde.csv` (méthode `load`) :
aucun appel aux API Reddit ou Arxiv.

In [1]:
from pathlib import Path

from Corpus import Corpus

corpus = Corpus("FIFA World Cup")
corpus.load(Path("data") / "corpus_sauvegarde.csv")
corpus

Corpus(nom='FIFA World Cup', 172 documents, 134 auteurs)

### 1.2 et 1.3 Instanciation du moteur
`SearchEngine(corpus)` construit aussitôt le vocabulaire et les matrices TF et TF-IDF.
Les barres de progression `tqdm` (1.3) affichent l'avancement de la construction.

In [2]:
from SearchEngine import SearchEngine

moteur = SearchEngine(corpus)
print(f"{moteur.mat_TF.shape[0]} documents x {moteur.mat_TF.shape[1]} mots")

Nettoyage des textes:   0%|          | 0/172 [00:00<?, ?it/s]

Matrice TF:   0%|          | 0/172 [00:00<?, ?it/s]

172 documents x 3932 mots


Vérification rapide : deux requêtes en mots-clés, comme en Partie 2.

In [3]:
moteur.search("messi argentina", 3)

,id,score,type,titre,auteur,date,url
0,63,0.406,reddit,Lionel Messi retires from Argentina internatio...,TheAthletic,31/08/2026,https://www.reddit.com/r/worldcup/comments/1w3...
1,24,0.160,reddit,Kylian Mbappé reaches Miroslav Klose’s goal re...,Swiftvoyager1906,23/06/2026,https://www.reddit.com/r/worldcup/comments/1ud...
2,58,0.152,reddit,Messi almost catches Vozinha off guard but the...,JCHazard,03/07/2026,https://www.reddit.com/r/worldcup/comments/1um...


In [4]:
moteur.search("cape verde", 3)

,id,score,type,titre,auteur,date,url
0,58,0.499,reddit,Messi almost catches Vozinha off guard but the...,JCHazard,03/07/2026,https://www.reddit.com/r/worldcup/comments/1um...
1,26,0.287,reddit,A journalist was interviewing a Cabo Verde fan...,GiveMeSomeSunshine3,04/07/2026,https://www.reddit.com/r/worldcup/comments/1un...
2,35,0.272,reddit,Post-Match Thread: Argentina 3-2 Cape Verde Is...,matchpal-live,04/07/2026,https://www.reddit.com/r/worldcup/comments/1um...


## 2. Interface graphique avec ipywidgets

### 2.1 Les objets graphiques simples
- un `Label` pour le titre ;
- un `Text` pour saisir les mots-clés ;
- un `IntSlider` (curseur d'entiers) pour choisir le nombre de documents à retourner.

In [5]:
import ipywidgets as widgets
from IPython.display import display

titre = widgets.Label(value="Moteur de recherche")
# Un Label n'accepte pas de HTML : on agrandit le texte par son style
titre.style.font_size = "24px"
titre.style.font_weight = "bold"

champ_mots_cles = widgets.Text(
    description="Mots clés",
    placeholder="ex. : messi argentina",          # texte grisé tant que le champ est vide
    style={"description_width": "initial"},       # l'étiquette n'est pas tronquée
)

curseur_nombre = widgets.IntSlider(
    description="Nombre d'articles à extraire",
    value=5, min=1, max=20,                       # 5 par défaut, entre 1 et 20
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),         # curseur plus large
)

### 2.3 La zone de résultats
Un `Output` : une zone qui recevra tout ce que la recherche affiche.

In [6]:
zone_resultats = widgets.Output()

### 2.5 La fonction `clique_bouton`
Elle lit les widgets (attribut `.value`), lance la recherche et affiche le résultat dans la
zone `Output` grâce à l'instruction `with`. La zone est vidée à chaque appel
(`clear_output`) : une nouvelle recherche remplace la précédente au lieu de s'y ajouter.
Cela neutralise aussi un comportement observé dans VS Code, qui transmet parfois un même
clic deux fois : le second appel efface l'affichage du premier.
Les cas particuliers (champ vide, aucun résultat) affichent un message.

In [7]:
def lien_cliquable(url: str) -> str:
    """Transforme une adresse en lien HTML qui s'ouvre dans un nouvel onglet."""
    return f'<a href="{url}" target="_blank">ouvrir</a>'


def clique_bouton(bouton: widgets.Button) -> None:
    """Appelée à chaque clic sur « Rechercher » (le bouton est passé en paramètre)."""
    mots_cles = champ_mots_cles.value.strip()      # strip : espaces au bord ignorés
    nombre = curseur_nombre.value

    # Tout ce qui est affiché dans ce bloc « with » va dans la zone de résultats
    with zone_resultats:
        zone_resultats.clear_output()      # on efface la recherche précédente

        if not mots_cles:                  # champ vide (ou seulement des espaces)
            print("Veuillez saisir au moins un mot-clé.")
            return

        resultats = moteur.search(mots_cles, nombre)
        if resultats.empty:                # aucun mot de la requête dans le corpus
            print(f"Aucun document ne contient les mots « {mots_cles} ».")
            return

        print(f"{len(resultats)} document(s) trouvé(s) pour « {mots_cles} » :")
        # Styler pandas : tableau sans index, score à 3 décimales,
        # URL transformée en lien cliquable
        display(resultats.style
                .format({"url": lien_cliquable, "score": "{:.3f}"})
                .hide(axis="index"))

### 2.4 Le bouton « Rechercher » et son branchement
`on_click` **ajoute** la fonction à la liste de celles que le bouton appelle. Le bouton est
donc créé **dans la même cellule** que son branchement : relancer cette cellule crée un
bouton neuf, branché une seule fois (sinon la fonction serait branchée une fois de plus).

In [8]:
bouton_rechercher = widgets.Button(
    description="Rechercher",
    button_style="primary",      # bouton bleu
    icon="search",               # icône de loupe
)

# On passe la fonction elle-même, SANS parenthèses : c'est le bouton qui l'appellera
bouton_rechercher.on_click(clique_bouton)

### 2.2 Assemblage et affichage de l'interface
`VBox` empile les éléments de haut en bas ; `HBox` (boîte horizontale) place le champ et le
bouton côte à côte.

In [9]:
interface = widgets.VBox([
    titre,
    widgets.HBox([champ_mots_cles, bouton_rechercher]),
    curseur_nombre,
    zone_resultats,
])
display(interface)

### 2.6 Vérification

**Comparaison avec la ligne de commande.** La recherche « cape verde » (5 articles) lancée
dans `main_partie2.py` (Partie 2) a donné :

| id | score | titre (début) |
|---|---|---|
| 58 | 0.499 | Messi almost catches Vozinha off guard… |
| 26 | 0.287 | A journalist was interviewing a Cabo Verde fan… |
| 35 | 0.272 | Post-Match Thread: Argentina 3-2 Cape Verde Islands… |
| 66 | 0.254 | Post-Match Thread: Spain 0-0 Cape Verde Islands… |
| 8 | 0.246 | Cape Verde coach Bubista has made it a tradition… |

La même recherche dans l'interface (et dans la cellule ci-dessous) doit donner exactement
le même classement : l'interface ne fait qu'appeler `moteur.search`.

In [10]:
moteur.search("cape verde", 5)[["id", "score", "titre"]]

,id,score,titre
0,58,0.499,Messi almost catches Vozinha off guard but the...
1,26,0.287,A journalist was interviewing a Cabo Verde fan...
2,35,0.272,Post-Match Thread: Argentina 3-2 Cape Verde Is...
3,66,0.254,Post-Match Thread: Spain 0-0 Cape Verde Island...
4,8,0.246,Cape Verde coach Bubista has made it a traditi...


**Majuscules et ponctuation.** La requête est nettoyée comme les documents
(`nettoyer_texte`) : « CAPE Verde !! » doit donner le même résultat que « cape verde ».

In [11]:
moteur.search("CAPE Verde !!", 5).equals(moteur.search("cape verde", 5))

True

**Cas particuliers testés dans l'interface**

| Saisie | Résultat attendu |
|---|---|
| champ vide, ou seulement des espaces | « Veuillez saisir au moins un mot-clé. » |
| mot absent du corpus (`ronaldo`, `cr7 portugal`) | « Aucun document ne contient les mots… » |
| plusieurs clics de suite | la nouvelle recherche remplace la précédente (`clear_output`) |
| curseur au minimum (1) puis au maximum (20) | 1 document, puis au plus 20 (moins si peu de documents contiennent les mots) |
| clic sur un lien « ouvrir » | la page Reddit ou Arxiv d'origine s'ouvre dans un nouvel onglet |